In [ ]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
from typing import Optional, List, Tuple

import pandas as pd


In [ ]:
# Question 1
def extract_infobox_value(infobox, label_text):
    row = infobox.find("th", string=lambda s: s and label_text.lower() in s.lower())
    if not row:
        return None
    td = row.find_next_sibling("td")
    if not td:
        return None
    return " ".join(td.get_text(" ", strip=True).split())


def get_country_info(url: str) -> dict:
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    if not infobox:
        return {
            "capital": None,
            "currency": None,
            "flag_url": None,
        }

    capital = extract_infobox_value(infobox, "Capital")
    currency = extract_infobox_value(infobox, "Currency")

    flag_url = None
    img = infobox.find("img")
    if img and img.get("src"):
        src = img.get("src")
        flag_url = src if src.startswith("http") else f"https:{src}"

    return {
        "capital": capital,
        "currency": currency,
        "flag_url": flag_url,
    }


def print_country_info(url: str, name: Optional[str] = None) -> None:
    info = get_country_info(url)
    if name:
        print(f"{name}:")
    print(f"Capital: {info['capital']}")
    print(f"Currency: {info['currency']}")
    print(f"Flag URL: {info['flag_url']}")
    print()


In [ ]:
print_country_info("https://en.wikipedia.org/wiki/Israel", "Israel")


In [ ]:
# Question 2
countries = [
    ("Italy", "https://en.wikipedia.org/wiki/Italy"),
    ("France", "https://en.wikipedia.org/wiki/France"),
    ("Spain", "https://en.wikipedia.org/wiki/Spain"),
    ("Algeria", "https://en.wikipedia.org/wiki/Algeria"),
    ("Australia", "https://en.wikipedia.org/wiki/Australia"),
]

for name, url in countries:
    print_country_info(url, name)


In [ ]:
# Question 3
def parse_city_from_location(text: str) -> Optional[str]:
    if not text:
        return None
    cleaned = " ".join(text.split())
    cleaned = cleaned.split("[")[0]
    cleaned = cleaned.split("(")[0].strip()
    if "," in cleaned:
        return cleaned.split(",")[0].strip()
    return cleaned.strip() if cleaned.strip() else None


def extract_university_city(url: str) -> Optional[str]:
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    if not infobox:
        return None

    row = infobox.find("th", string=lambda s: s and "Location" in s)
    if not row:
        return None
    td = row.find_next_sibling("td")
    if not td:
        return None

    location_text = " ".join(td.get_text(" ", strip=True).split())
    return parse_city_from_location(location_text)


def collect_university_links(list_url: str) -> List[Tuple[str, str]]:
    resp = requests.get(list_url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    content = soup.find("div", class_="mw-parser-output")
    if not content:
        return []

    links = []
    seen = set()

    for a in content.find_all("a", href=True):
        href = a["href"]
        title = a.get_text(strip=True)
        if not title or not href.startswith("/wiki/"):
            continue
        if ":" in href:
            continue
        full_url = f"https://en.wikipedia.org{href}"
        if full_url in seen:
            continue
        seen.add(full_url)
        links.append((title, full_url))

    return links


list_url = "https://en.wikipedia.org/wiki/List_of_universities_in_Germany"
university_links = collect_university_links(list_url)

city_to_unis = defaultdict(list)

for name, url in university_links:
    city = extract_university_city(url)
    if city:
        city_to_unis[city].append(name)

for city, unis in sorted(city_to_unis.items()):
    if len(unis) > 2:
        print(f"{city}:")
        for uni in sorted(set(unis)):
            print(f"- {uni}")
        print()
